# Ψ₀ policy server trên Colab, mô phỏng MuJoCo trên máy local

Notebook chạy checkpoint Ψ₀ đã finetune (từ `colab_finetune_psi0.ipynb`) trên GPU Colab, phục vụ qua HTTP (`serve_psi0_sonic_http`, `POST /act`),
và mở đường hầm TCP Pinggy để máy local gọi tới. Mô phỏng MuJoCo (scene box_pnp, SONIC, cửa sổ viewer) chạy trên máy local bằng
`scripts/datagen/box_pnp/closed_loop_eval.py`.

**Yêu cầu:** checkpoint nằm trên Google Drive của tài khoản chạy Colab, ở `My Drive/<DRIVE_RUNS>/finetune/...` (notebook finetune ghi vào đó).
GPU **L4, A100, H100 hoặc G4**. Không cần dataset: server chỉ cần thư mục run (`argv.txt`, `run_config.json`, `checkpoints/ckpt_N`).

| Cell | Việc |
|---|---|
| 1 | Cấu hình: run (mặc định run mới nhất của `EXP`) và checkpoint (mặc định checkpoint mới nhất trên Drive) |
| 2 | GPU, kết nối Drive (hiện hộp thoại cấp quyền), chọn checkpoint |
| 3 | Cài môi trường (giống notebook finetune), 10–15 phút |
| 4 | Ghi `.env` |
| 5 | Tải thư mục run và checkpoint từ Drive (~10 GB) |
| 6 | Chạy server nền, chờ sẵn sàng, gọi thử một lần |
| 7 | Mở đường hầm Pinggy, in lệnh chạy trên máy local |
| 8 | Trạng thái server |
| 9 | Dừng server (đặt `STOP_SERVER = True`) |

**Trên máy local** (sau cell 7), từ thư mục repo:
```
.venv-datagen/Scripts/python scripts/datagen/box_pnp/closed_loop_eval.py --server http://HOST:PORT --episodes 10 --viewer --out .data/eval/<tên>
```
- `--viewer` mở cửa sổ MuJoCo (góc nhìn thứ ba; chuột phải/trái để xoay, kéo). Mô phỏng tạm dừng trong lúc chờ server, nên kết quả không phụ thuộc độ trễ mạng.
- `--video` ghi video ego và góc thứ ba của từng scene vào `--out`. Kết quả từng scene nằm ở `<out>/results.jsonl`.
- Các scene dùng seed từ 1 000 000, chưa xuất hiện trong dữ liệu train.
- Kiểm tra phía mô phỏng mà không cần model: `--replay .data/box_pnp_tray_140 --replay-episode 0 --viewer`.

Tunnel Pinggy miễn phí hết hạn sau khoảng 60 phút: chạy lại cell 7 để lấy host/port mới, server không cần khởi động lại.

In [ ]:
# @title 1. Cấu hình
# Only declares the settings: cell 2 exports them for the other cells, so after editing this cell run cell 2 as well.
CFG = {
    "PSI0_REPO": "https://github.com/physical-superintelligence-lab/Psi0.git",
    "PSI0_COMMIT": "4f3720d45e102b36d7c3e9465ab8062274170518",
    "DRIVE_RUNS": "psi0/runs",  # on the Google Drive of the account running this notebook, relative to My Drive
    "EXP": "boxpnp",
    "RUN_TS": "",               # empty: the run id in <DRIVE_RUNS>/<EXP>.run_ts (the finetune notebook's run)
    "CKPT_STEP": "",            # empty: the newest checkpoint of that run on Drive
    "PORT": "8014",
}

In [ ]:
# @title 2. GPU, Google Drive, checkpoint
import io
import os
import pathlib
import re
import subprocess

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

os.environ["NB_READY"] = ""  # set again at the end, once everything below is exported
os.environ.update(CFG)
os.environ["PSI_HOME"] = "/content/psi_home"

name, cc = [s.strip() for s in subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True).splitlines()[0].split(",")]
print(f"GPU: {name}, compute capability {cc}")
if float(cc) < 8.0:
    raise RuntimeError("Need an Ampere or newer GPU (L4 / A100 / H100 / G4): the model runs in bf16.")


class Drive:
    """My Drive of the account running this notebook, through the Drive API.

    Not drive.mount: a file deleted through the mount goes to the Drive trash and keeps counting
    against the quota (10 GB per rotated checkpoint), and the mount uploads in the background, so a
    checkpoint could look complete on Drive before all of it is there. Here upload() returns once
    the bytes are on Drive, and delete() is permanent.
    """
    FOLDER = "application/vnd.google-apps.folder"

    def __init__(self):
        auth.authenticate_user()
        self.svc = build("drive", "v3", cache_discovery=False)

    def children(self, folder_id):
        out, token = [], None
        while True:
            r = self.svc.files().list(q=f"'{folder_id}' in parents and trashed = false", pageSize=1000,
                                      pageToken=token, fields="nextPageToken, files(id, name, mimeType, size)"
                                      ).execute(num_retries=5)
            out += r["files"]
            token = r.get("nextPageToken")
            if not token:
                return out

    def child(self, folder_id, name):
        return next((f for f in self.children(folder_id) if f["name"] == name), None)

    def mkdir(self, parent, name):
        return self.svc.files().create(body={"name": name, "mimeType": self.FOLDER, "parents": [parent]},
                                       fields="id").execute(num_retries=5)["id"]

    def path(self, rel, create=False):
        """Id of the file or folder at `rel` (relative to My Drive), or None."""
        fid = "root"
        for part in pathlib.PurePosixPath(rel).parts:
            f = self.child(fid, part)
            if f is None and not create:
                return None
            fid = f["id"] if f else self.mkdir(fid, part)
        return fid

    def upload(self, local, parent, name=None):
        """Upload a file, or a directory recursively, into `parent`; replaces a file of the same name."""
        local = pathlib.Path(local)
        name = name or local.name
        if local.is_dir():
            fid = self.mkdir(parent, name)
            for p in sorted(local.iterdir()):
                if not p.is_symlink():
                    self.upload(p, fid)
            return fid
        size = local.stat().st_size
        old = self.child(parent, name)
        if size == 0:  # nothing to send; an empty upload body is rejected
            return old["id"] if old else self.svc.files().create(
                body={"name": name, "parents": [parent]}, fields="id").execute(num_retries=5)["id"]
        media = MediaFileUpload(str(local), resumable=size > 8 * 2**20, chunksize=256 * 2**20)
        if old:
            req = self.svc.files().update(fileId=old["id"], media_body=media, fields="id, size")
        else:
            req = self.svc.files().create(body={"name": name, "parents": [parent]}, media_body=media,
                                          fields="id, size")
        if media.resumable():
            resp = None
            while resp is None:
                _, resp = req.next_chunk(num_retries=5)
        else:
            resp = req.execute(num_retries=5)
        if int(resp.get("size", -1)) != size:
            raise IOError(f"{local}: Drive holds {resp.get('size')} bytes, expected {size}")
        return resp["id"]

    def download(self, file_id, local):
        local = pathlib.Path(local)
        local.parent.mkdir(parents=True, exist_ok=True)
        with open(local, "wb") as fh:
            dl = MediaIoBaseDownload(fh, self.svc.files().get_media(fileId=file_id), chunksize=64 * 2**20)
            done = False
            while not done:
                _, done = dl.next_chunk(num_retries=5)

    def download_tree(self, item, local):
        """`item`: an entry of children(); a folder is downloaded recursively."""
        if item["mimeType"] == self.FOLDER:
            for c in self.children(item["id"]):
                self.download_tree(c, pathlib.Path(local) / c["name"])
        else:
            self.download(item["id"], local)

    def read_text(self, rel):
        fid = self.path(rel)
        if fid is None:
            return ""
        buf = io.BytesIO()
        dl = MediaIoBaseDownload(buf, self.svc.files().get_media(fileId=fid))
        done = False
        while not done:
            _, done = dl.next_chunk(num_retries=5)
        return buf.getvalue().decode()

    def write_text(self, rel, text):
        rel = pathlib.PurePosixPath(rel)
        tmp = pathlib.Path("/tmp") / rel.name
        tmp.write_text(text)
        self.upload(tmp, self.path(str(rel.parent), create=True))

    def rename(self, file_id, name):
        self.svc.files().update(fileId=file_id, body={"name": name}).execute(num_retries=5)

    def delete(self, file_id):
        self.svc.files().delete(fileId=file_id).execute(num_retries=5)  # permanent: skips the trash

    def free_bytes(self):
        q = self.svc.about().get(fields="storageQuota").execute(num_retries=5)["storageQuota"]
        return int(q["limit"]) - int(q["usage"]) if "limit" in q else None



gd = Drive()
run_ts = CFG["RUN_TS"] or gd.read_text(f"{CFG['DRIVE_RUNS']}/{CFG['EXP']}.run_ts").strip()
assert run_ts, f"no run id: My Drive/{CFG['DRIVE_RUNS']}/{CFG['EXP']}.run_ts not found - set RUN_TS"
finetune = gd.path(f"{CFG['DRIVE_RUNS']}/finetune")
run = next((f for f in (gd.children(finetune) if finetune else [])
            if f["mimeType"] == gd.FOLDER and f["name"].endswith(f".{run_ts}")), None)
assert run, f"no run folder ending in .{run_ts} under My Drive/{CFG['DRIVE_RUNS']}/finetune"
ck = gd.child(run["id"], "checkpoints")
on_drive = {int(m.group(1)): f for f in (gd.children(ck["id"]) if ck else [])
            if (m := re.fullmatch(r"ckpt_(\d+)", f["name"]))}
assert on_drive, f"{run['name']} has no complete checkpoint on Drive yet"
step = int(CFG["CKPT_STEP"]) if CFG["CKPT_STEP"] else max(on_drive)
assert step in on_drive, f"ckpt_{step} is not on Drive; there: {sorted(on_drive)}"
os.environ.update(RUN_DIR=f"/content/runs/finetune/{run['name']}", CKPT_STEP=str(step))
print(f"run {run['name']}\ncheckpoint ckpt_{step} (on Drive: {sorted(on_drive)})")
os.environ["NB_READY"] = "1"

In [ ]:
%%bash
# 3. Environment: /content/Psi0 at PSI0_COMMIT + .venv-psi (Python 3.11, torch 2.7, flash-attn or SDPA)
set -euo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
# Colab's own values would leak into the venv; UV_SYSTEM_PYTHON would point uv at Colab's python
unset PYTHONPATH MPLBACKEND UV_SYSTEM_PYTHON
export GIT_LFS_SKIP_SMUDGE=1 PATH="$HOME/.local/bin:$PATH"
# uv.lock at PSI0_COMMIT does not parse: its relock (6b4a5cc) left a duplicated table and packages
# listed under two registries. This is the last lock that parses, all on PyPI; the psi group is the
# same since, and the only newer dependencies (decord2, wandb[media]) are not used by this training.
GOOD_LOCK_COMMIT=47993ee31926235b2a640cc554a8c93f66220a3a
# the wheel scripts/train/Dockerfile uses: cu12 / torch 2.7 / cxx11abi TRUE / cp311, sm80 + sm90 kernels
FLASH_ATTN_WHEEL="https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"
# Blackwell (G4 = RTX PRO 6000, compute capability 12.0): the locked torch 2.7.0 is the CUDA 12.6 build,
# which has no sm_100/sm_120 kernels, and flash-attn 2 publishes no Blackwell wheel. There the same
# torch 2.7.0 comes from the CUDA 12.8 index (README troubleshooting 5) and the VLM uses PyTorch's
# SDPA attention: the same attention, computed by a different kernel.
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n 1)
if [ "${CC%%.*}" -ge 10 ]; then VARIANT=cu128-sdpa; else VARIANT=cu126-flash; fi
echo "GPU compute capability $CC -> $VARIANT"

mkdir -p /content/Psi0 && cd /content/Psi0
[ -d .git ] || { git init -q && git remote add origin "$PSI0_REPO"; }
if [ "$(git rev-parse -q --verify HEAD || true)" != "$PSI0_COMMIT" ]; then
    git fetch -q --depth 1 origin "$PSI0_COMMIT"
    git checkout -q --force FETCH_HEAD
fi
echo "Psi0 @ $(git rev-parse --short HEAD)"

command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh
if [ -f .venv-psi/.synced ]; then
    echo "locked dependencies already installed"
else
    uv python install 3.11
    [ -d .venv-psi ] || uv venv .venv-psi --python 3.11
    source .venv-psi/bin/activate
    git fetch -q --depth 1 origin "$GOOD_LOCK_COMMIT"
    git show FETCH_HEAD:uv.lock > uv.lock
    # --frozen: install the lock as-is; --inexact: keep what is installed on top of it (flash-attn,
    # the CUDA 12.8 torch)
    uv sync --active --frozen --group psi --inexact
    touch .venv-psi/.synced
fi
# Checked against what is installed rather than a marker, so a half-finished run repairs itself.
if [ "$VARIANT" = cu128-sdpa ]; then
    if [ "$(.venv-psi/bin/python -c 'import torch; print(torch.version.cuda)')" != 12.8 ]; then
        # the +cu128 local version is required: a plain ==2.7.0 is already satisfied by the
        # locked 2.7.0+cu126, and uv would keep that build
        uv pip install --python .venv-psi/bin/python torch==2.7.0+cu128 torchvision==0.22.0+cu128 \
            --index-url https://download.pytorch.org/whl/cu128
    fi
elif ! .venv-psi/bin/python -c "import flash_attn" 2>/dev/null; then
    uv pip install --python .venv-psi/bin/python "$FLASH_ATTN_WHEEL"
fi
# the HTTP server needs the serve group (fastapi, uvicorn), pinned to the lock's versions. Installed
# by itself: a uv sync would put back the locked CUDA 12.6 torch over the 12.8 one.
if ! .venv-psi/bin/python -c "import fastapi, uvicorn" 2>/dev/null; then
    uv pip install --python .venv-psi/bin/python fastapi==0.119.1 uvicorn==0.38.0
fi
if [ "$VARIANT" = cu128-sdpa ]; then
    # finetune.py hard-codes flash_attention_2 for the VLM (the serving code already falls back to sdpa)
    sed -i 's/attn_implementation="flash_attention_2"/attn_implementation="sdpa"/' src/psi/trainers/finetune.py
fi
grep -n 'attn_implementation=' src/psi/trainers/finetune.py

source .venv-psi/bin/activate
python - "$VARIANT" <<'PY'
import sys, torch, psi
from psi.data.lerobot.compat import LEROBOT_LAYOUT
# fails here, not mid-training, when the build has no kernel for this GPU
x = torch.randn(256, 256, device="cuda", dtype=torch.bfloat16)
(x @ x).sum().item()
if sys.argv[1] == "cu126-flash":
    import flash_attn
    from flash_attn import flash_attn_func
    q = torch.randn(1, 64, 4, 64, device="cuda", dtype=torch.bfloat16)
    flash_attn_func(q, q, q)
    attn = f"flash_attn {flash_attn.__version__}"
else:
    q = torch.randn(1, 4, 64, 64, device="cuda", dtype=torch.bfloat16)
    torch.nn.functional.scaled_dot_product_attention(q, q, q).sum().item()
    attn = "sdpa"
print(f"torch {torch.__version__} (cuda {torch.version.cuda}) | {attn} | lerobot layout {LEROBOT_LAYOUT} | "
      f"{torch.cuda.get_device_name(0)}")
PY

In [ ]:
%%bash
# 4. .env: scripts/train.py asserts load_dotenv() finds one. Secrets stay in the process
# environment (cell 2) and are never written to disk.
set -euo pipefail
: "${NB_READY:?run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)}"
mkdir -p "$PSI_HOME"/data "$PSI_HOME"/cache/hf "$PSI_HOME"/cache/checkpoints
cat > /content/Psi0/.env <<EOF
PSI_HOME=$PSI_HOME
DATA_HOME=$PSI_HOME/data
HF_HOME=$PSI_HOME/cache/hf
TORCH_HOME=$PSI_HOME/cache/hf
HF_LEROBOT_HOME=$PSI_HOME/data/lerobot
OMP_NUM_THREADS=8
TOKENIZERS_PARALLELISM=false
CUDA_LAUNCH_BLOCKING=0
TF_CPP_MIN_LOG_LEVEL=3
AV_LOG_LEVEL=quiet
PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
NO_ALBUMENTATIONS_UPDATE=1
EOF
cat /content/Psi0/.env

In [ ]:
# @title 5. Tải thư mục run và checkpoint từ Drive
import os
import pathlib

assert os.environ.get("NB_READY"), "run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)"
run_dir = pathlib.Path(os.environ["RUN_DIR"])


def fetch(item, local):
    """Download a Drive file or folder unless an identical-size local copy exists."""
    local = pathlib.Path(local)
    if item["mimeType"] == gd.FOLDER:
        for c in gd.children(item["id"]):
            fetch(c, local / c["name"])
    elif not (local.exists() and local.stat().st_size == int(item.get("size", -1))):
        print(f"  {local.relative_to(run_dir.parent)} ({int(item.get('size', 0)) / 2**30:.2f} GB)")
        gd.download(item["id"], local)


for f in gd.children(run["id"]):
    if f["mimeType"] != gd.FOLDER:
        fetch(f, run_dir / f["name"])
fetch(on_drive[step], run_dir / "checkpoints" / f"ckpt_{step}")
print("ready:", run_dir, "| ckpt", step)

In [ ]:
# @title 6. Chạy server nền và gọi thử
# Bấm ■ chỉ dừng việc chờ, server vẫn chạy; xem trạng thái ở cell 8, dừng ở cell 9.
import json
import os
import subprocess
import time
import urllib.request
from base64 import b64encode

import numpy as np
from numpy.lib.format import dtype_to_descr

assert os.environ.get("NB_READY"), "run cell 2 first (after a runtime restart or an edit of cell 1, re-run cells 1 and 2)"
PORT = os.environ["PORT"]
URL = f"http://127.0.0.1:{PORT}"
SERVER_LOG = "/content/server.log"


def healthy():
    try:
        with urllib.request.urlopen(f"{URL}/health", timeout=2) as r:
            return json.loads(r.read()).get("status") == "ok"
    except Exception:
        return False


if healthy():
    print(f"a server already answers on port {PORT} (stop it with cell 9 to change the checkpoint)")
else:
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "MPLBACKEND")}
    server = subprocess.Popen(
        ["/content/Psi0/.venv-psi/bin/serve_psi0_sonic_http", "--policy", "psi0", "--run-dir", os.environ["RUN_DIR"],
         "--ckpt-step", os.environ["CKPT_STEP"], "--host", "0.0.0.0", "--port", PORT],
        cwd="/content/Psi0", stdout=open(SERVER_LOG, "w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        env=env, start_new_session=True)
    print(f"server starting (PID {server.pid}), log {SERVER_LOG} ...")
    t0 = time.time()
    while not healthy():
        if server.poll() is not None:
            print(open(SERVER_LOG, errors="replace").read()[-4000:])
            raise RuntimeError(f"server exited with code {server.returncode}, see the log above")
        time.sleep(5)
    print(f"server up after {time.time() - t0:.0f} s")


# one request in the client's format, so the first real one does not pay for kernel warm-up
def enc(a):
    a = np.ascontiguousarray(a)
    return {"__numpy__": b64encode(a.tobytes()).decode(), "dtype": dtype_to_descr(a.dtype), "shape": list(a.shape)}


payload = {"image": {"observation.images.egocentric": enc(np.zeros((240, 320, 3), np.uint8))},
           "instruction": "warm up", "history": {}, "state": {"states": enc(np.zeros(43, np.float32))},
           "condition": {}, "gt_action": [], "dataset_name": "box_pnp", "timestamp": str(time.time())}
for i in range(2):
    t0 = time.time()
    req = urllib.request.Request(f"{URL}/act", data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    body = json.loads(urllib.request.urlopen(req, timeout=300).read())
    assert "action" in body, f"server error: {body}"
    print(f"test request {i + 1}: action {body['action']['shape']} in {time.time() - t0:.2f} s")

In [ ]:
# @title 7. Mở đường hầm TCP Pinggy và in lệnh cho máy local
# Tunnel miễn phí hết hạn sau khoảng 60 phút: chạy lại cell này để lấy host/port mới (server không cần khởi động lại).
import os
import subprocess
import time

if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    print("closed the previous tunnel")
tunnel_proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "-p", "443",
     f"-R0:localhost:{os.environ['PORT']}", "tcp@free.pinggy.io"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
pub = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if "tcp://" in line:
        pub = line.strip().split("tcp://")[1].split()[0].rstrip("/")
        break
    if not line:
        time.sleep(0.5)
if not pub:
    raise RuntimeError("could not read the tunnel address; run this cell again")
host, port = pub.split(":")
print("=" * 90)
print(f"tunnel http://{host}:{port} (opened {time.strftime('%H:%M')} UTC, expires after ~60 min)")
print("=" * 90)
print("On the local machine, from the repo root:")
print(f".venv-datagen/Scripts/python scripts/datagen/box_pnp/closed_loop_eval.py --server http://{host}:{port} "
      f"--episodes 10 --viewer --out .data/eval/ckpt{os.environ['CKPT_STEP']}")
print("=" * 90)

In [ ]:
# @title 8. Trạng thái server
import os
import re
import urllib.request

try:
    up = b'"ok"' in urllib.request.urlopen(f"http://127.0.0.1:{os.environ['PORT']}/health", timeout=2).read()
except Exception:
    up = False
log = open("/content/server.log", errors="replace").read() if os.path.exists("/content/server.log") else ""
print("server:", "answering on port " + os.environ["PORT"] if up else "NOT running")
print("tunnel:", "open" if "tunnel_proc" in globals() and tunnel_proc.poll() is None else "closed (run cell 7)")
print(f"requests served: {len(re.findall('Return Action', log))} | tracebacks in the log: {log.count('Traceback')}")
print("--- end of the log")
print("\n".join(re.split(r"[\r\n]+", log.strip())[-15:]))

In [ ]:
# @title 9. Dừng server
# Only with STOP_SERVER = True, so "Run all" does not stop the server cell 6 just started.
import subprocess
import time

STOP_SERVER = False
if not STOP_SERVER:
    print("STOP_SERVER is False: nothing stopped")
else:
    subprocess.run(["pkill", "-f", "serve_psi0_sonic_http"])
    time.sleep(3)
    print("server stopped" if subprocess.run(["pgrep", "-f", "serve_psi0_sonic_http"], capture_output=True).returncode
          else "server still running; run this cell again")